# Customer Churn Notebook

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config.configuration import AppConfig
from src.pipelines.portfolio_showcase_pipeline import run_portfolio_showcase_pipeline

sns.set_theme(style='whitegrid')
assets_dir = project_root / 'assets'
assets_dir.mkdir(parents=True, exist_ok=True)

config = AppConfig.from_env()
df = pd.read_csv(project_root / 'data' / 'raw' / 'telco_churn.csv')
df.head()

In [ ]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
display(df.describe(include='all').T)

churn_dist = df['Churn'].astype(str).value_counts().rename_axis('Churn').to_frame('count')
churn_dist['ratio'] = (churn_dist['count'] / churn_dist['count'].sum()).round(4)
display(churn_dist)

In [ ]:
# EDA distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
class_counts = df['Churn'].astype(str).value_counts()
labels = class_counts.index.astype(str)
sns.barplot(x=labels, y=class_counts.values, hue=labels, legend=False, palette='Set2', ax=axes[0])
axes[0].set_title('Churn Class Distribution')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')

sns.histplot(data=df, x='MonthlyCharges', hue='Churn', bins=30, kde=True, element='step', stat='density', common_norm=False, ax=axes[1])
axes[1].set_title('Monthly Charges by Churn Class')
axes[1].set_xlabel('MonthlyCharges')

fig.tight_layout()
fig.savefig(assets_dir / 'eda_distribution.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

# Missing values
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=missing_pct.index.astype(str), y=missing_pct.values, color='#4C78A8', ax=ax)
ax.set_title('Missing Values by Feature (%)')
ax.set_ylabel('Missing %')
ax.set_xlabel('Feature')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
fig.savefig(assets_dir / 'missing_values.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

# Correlation heatmap
corr_df = df.copy()
corr_df['Churn'] = corr_df['Churn'].astype(str).str.lower().map({'yes': 1, 'no': 0, '1': 1, '0': 0})
num_df = corr_df.select_dtypes(include=['number', 'bool'])
corr = num_df.corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True, fmt='.2f', linewidths=0.4, ax=ax)
ax.set_title('Numeric Feature Correlation')
fig.tight_layout()
fig.savefig(assets_dir / 'correlation_heatmap.png', dpi=140, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
# Run full showcase generation (model visuals + API visual + real outputs)
manifest = run_portfolio_showcase_pipeline(config)
manifest

In [ ]:
with (project_root / 'artifacts' / 'metrics.json').open('r', encoding='utf-8') as f:
    metrics = json.load(f)
with (project_root / 'artifacts' / 'output_samples.json').open('r', encoding='utf-8') as f:
    outputs = json.load(f)

display(pd.Series(metrics.get('metrics', {}), name='value').to_frame())
outputs

## Churn Insights

- Churn is strongly associated with contract type, support availability, and monthly bill levels.
- Short tenure plus month-to-month profiles contribute disproportionately to high-risk predictions.
- The showcase pipeline creates all visuals and real output examples automatically for portfolio presentation.